# RAG simples com Ollama (nomic-embed-text)

Este notebook lê uma uma pasta DATA_DIR com PDFs, faz chunking, gera embeddings com `nomic-embed-text` no Ollama, e insere no Chroma. Recupera exemplos de chunks mais similares para uma pergunta.

**Pré-requisito:** Ollama em execução em `http://localhost:11434` e modelos baixados (`nomic-embed-text` e um modelo de chat).

## 1) Instalar dependências


In [ ]:
!pip -q install requests numpy chromadb pypdf tqdm

## 2) Configurações


In [ ]:
import os
import json
import requests
import re
import uuid
import chromadb
from tqdm import tqdm
from pathlib import Path
from pypdf import PdfReader
from chromadb.config import Settings
from dotenv import load_dotenv, find_dotenv

# --------------------------------------
# Carrega variáveis do .env
# --------------------------------------
load_dotenv()

# --------------------------------------
# Configurações
# --------------------------------------

BATCH = int(os.getenv("BATCH", 64))

DATA_DIR = Path(os.getenv("DATA_DIR"))
PROJECT_ROOT = os.getenv("PROJECT_ROOT")

COLLECTION_NAME = os.getenv("COLLECTION_NAME", "base_pdfs")

# Chunking
CHUNK_SIZE = int(os.getenv("CHUNK_SIZE", 2048))
CHUNK_OVERLAP = int(os.getenv("CHUNK_OVERLAP", 150))
MIN_CHUNK_LEN = int(os.getenv("MIN_CHUNK_LEN", 200))

# Paths derivados
CHROMADB_PATH = f"chromadb_chunk_size_{CHUNK_SIZE}_overlap_{CHUNK_OVERLAP}"
INDEX_PATH = Path(PROJECT_ROOT) / CHROMADB_PATH

AUDIT_PATH = Path(PROJECT_ROOT) / f"chunks_audit_{CHUNK_SIZE}.json"

# Ollama
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
EMBED_MODEL = os.getenv("EMBED_MODEL", "nomic-embed-text")

# Retrieval
TOP_K = int(os.getenv("TOP_K", 5))

### 2.1) Verifica configurações

In [ ]:
print("Diretório atual:", os.getcwd())
dotenv_path = find_dotenv()
print("Arquivo .env encontrado:", dotenv_path)

vars_env = [
    "DATA_DIR",
    "PROJECT_ROOT",
    "COLLECTION_NAME",
    "CHUNK_SIZE",
    "CHUNK_OVERLAP",
    "MIN_CHUNK_LEN",
    "OLLAMA_BASE_URL",
    "EMBED_MODEL",
    "TOP_K",
    "BATCH"
]

print("\n===== TESTE DAS VARIÁVEIS DE AMBIENTE =====\n")

for v in vars_env:
    value = os.getenv(v)

    if value is None:
        print(f"{v:20} -> ❌ NÃO CARREGADA (None)")
    else:
        print(f"{v:20} -> ✅ {value}")

## 3) Funções: chamada ao Ollama (embeddings)


In [ ]:
def ollama_embed(text: str, model: str = EMBED_MODEL) -> list[float]:
    """Gera embedding via Ollama /api/embeddings."""
    url = f"{OLLAMA_BASE_URL}/api/embeddings"
    payload = {"model": model, "prompt": text}
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    data = r.json()
    return data["embedding"]


## 4) Ler PDFs da pasta e gerar chunks


In [ ]:
def clean_text(s: str) -> str:
    if not s:
        return ""
    # normaliza espaços
    s = re.sub(r"\s+", " ", s).strip()
    return s

def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP):
    """Chunking simples por caracteres com overlap."""
    text = clean_text(text)
    if len(text) < MIN_CHUNK_LEN:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        chunk = text[start:end].strip()
        if len(chunk) >= MIN_CHUNK_LEN:
            chunks.append(chunk)
        if end == len(text):
            break
        start = max(0, end - overlap)
    return chunks

In [ ]:
pdf_paths = sorted(DATA_DIR.glob("*.pdf"))
print(f"📄 PDFs encontrados: {len(pdf_paths)} em {DATA_DIR}")

docs = []  # lista de dicts: {id, text, metadata}

for pdf_path in tqdm(pdf_paths, desc="Lendo PDFs"):
    try:
        reader = PdfReader(str(pdf_path))
    except Exception as e:
        print(f"⚠️ Falha ao abrir {pdf_path.name}: {e}")
        continue

    for page_idx, page in enumerate(reader.pages, start=1):
        try:
            page_text = page.extract_text() or ""
        except Exception:
            page_text = ""

        for chunk_idx, chunk in enumerate(chunk_text(page_text), start=1):
            docs.append({
                "id": str(uuid.uuid4()),
                "text": chunk,
                "metadata": {
                    "source_pdf": pdf_path.name,
                    "page": page_idx,
                    "chunk": chunk_idx,
                }
            })

print(f"✅ Total de chunks gerados: {len(docs)}")


## 5) Criar/atualizar índice no ChromaDB (persistente)


In [ ]:
INDEX_DIR = Path(INDEX_PATH)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

settings = Settings(
    _env_file=None,               # << impede tentar ler ".env"
    anonymized_telemetry=False
)

client = chromadb.PersistentClient(path=str(INDEX_DIR), settings=settings)

collection = client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

print("📦 Coleção:", COLLECTION_NAME)
print("📁 Persistência em:", INDEX_DIR)
print("🔢 Itens atuais na coleção:", collection.count())


## 6) Gerar embeddings e inserir (com batch simples) no ChromaDB


In [ ]:
if not docs:
    raise RuntimeError("Não há chunks (docs) para inserir. Verifique DATA_DIR e o chunking.")

for i in tqdm(range(0, len(docs), BATCH), desc="Ingerindo no Chroma"):
    batch = docs[i:i+BATCH]
    ids = [d["id"] for d in batch]
    texts = [d["text"] for d in batch]
    metas = [d["metadata"] for d in batch]

    # embeddings via Ollama (um por chunk; simples e confiável)
    embeds = [ollama_embed(t) for t in texts]

    collection.add(
        ids=ids,
        documents=texts,
        metadatas=metas,
        embeddings=embeds,
    )

print(f"✅ Ingestão concluída. Total na coleção: {collection.count()}")

## 7) Consulta rápida (recuperação top-k)


In [ ]:
def chroma_retrieve(query: str, top_k: int = TOP_K):
    q_emb = ollama_embed(query)
    res = collection.query(
        query_embeddings=[q_emb],
        n_results=top_k,
        include=["documents", "metadatas", "distances"],
    )
    # organiza
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"distance": float(dist), "metadata": meta, "text": doc})
    return hits

query = "Qual é o valor total do contrato de pneus?"
hits = chroma_retrieve(query, top_k=5)

for i, h in enumerate(hits, start=1):
    m = h["metadata"]
    print(f"\n=== Hit {i} | dist={h['distance']:.4f} | {m['source_pdf']} | pág {m['page']} | chunk {m['chunk']} ===")
    print(h["text"][:900], "...")


## 8) Salvar chunks/metadata em JSON para auditoria


In [ ]:
with open(AUDIT_PATH, "w", encoding="utf-8") as f:
    json.dump(
        [{"id": d["id"], **d["metadata"], "text_preview": d["text"][:200]} for d in docs],
        f, ensure_ascii=False, indent=2
    )

print("📝 Auditoria salva em:", AUDIT_PATH)


In [ ]:
# Recuperar o primeiro item da coleção

data = collection.get(
    limit=1,
    include=["documents", "metadatas"]
)

print("ID:", data["ids"][0])
print("\nTexto do chunk:\n")
print(data["documents"][0][:500])

print("\nMetadata:")
print(data["metadatas"][0])

In [ ]:
# recuperar embedding do primeiro item

first_id = data["ids"][0]

emb = collection.get(
    ids=[first_id],
    include=["embeddings"]
)["embeddings"][0]

print("\nDimensão do embedding:", len(emb))
print("Primeiros valores:", emb[:10])

In [ ]:
del collection
del client